In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
import numpy as np 
import pandas as pd 
import os
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.metrics import log_loss
from datetime import datetime
import matplotlib.pyplot as plt

In [55]:
education_train = pd.read_csv('module_Education_train_set.csv')
education_test = pd.read_csv('module_Education_test_set.csv')
household_train = pd.read_csv('module_HouseholdInfo_train_set.csv')
household_test = pd.read_csv('module_HouseholdInfo_test_set.csv')
subjective_poverty_train = pd.read_csv('module_SubjectivePoverty_train_set.csv')
sample_submission = pd.read_csv('sample_submission.csv')
merged_data = pd.read_csv('merged_data1.csv')

C:\Users\Sadman\AppData\Local\Temp\ipykernel_42728\1688603801.py:7: DtypeWarning: Columns (61,146) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_data = pd.read_csv('merged_data1.csv')


In [14]:
target_columns = [f'subjective_poverty_{i}' for i in range(1, 11)]

In [ ]:
merged_train = merged_data.dropna(subset=target_columns)
X = merged_train.drop(columns=target_columns)
y = merged_train[target_columns]
X = X.drop(columns=X.select_dtypes(include=['object']).columns)
X = X.drop(columns=['hhid'])
y_class_labels = y.values.argmax(axis=1)



In [53]:
from sklearn.model_selection import KFold
from xgboost import XGBClassifier
from sklearn.metrics import log_loss
import numpy as np

# Function for cross-validation

def cross_validate_xgboost(X, y, y_class_labels, n_splits, reg_lambda, reg_alpha):
    cv_scores_val = []  # Validation scores
    cv_scores_train = []  # Training scores

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for fold, (train_index, val_index) in enumerate(kf.split(X), 1):
        # Split data into training and validation sets
        X_train, X_val = X.iloc[train_index], X.iloc[val_index]

        # Get class labels for training
        y_train = y_class_labels[train_index]

        # Get one-hot encoded labels for training and validation
        y_train_onehot = y.iloc[train_index].values
        y_val_onehot = y.iloc[val_index].values

        # Train XGBoost model
        xgb_model = XGBClassifier(
            objective='multi:softprob',  # Multiclass classification
            num_class=10,                # Number of classes
            eval_metric='mlogloss',      # Multiclass log loss
            random_state=42,
            reg_lambda=reg_lambda,
            reg_alpha=reg_alpha,
        )
        xgb_model.fit(X_train, y_train)

        # Predict probabilities for training and validation sets
        y_pred_proba_train = xgb_model.predict_proba(X_train)  # Training probabilities
        y_pred_proba_val = xgb_model.predict_proba(X_val)      # Validation probabilities

        # Compute multiclass log loss for training and validation
        train_loss = log_loss(y_train_onehot, y_pred_proba_train)
        val_loss = log_loss(y_val_onehot, y_pred_proba_val)

        # Append scores
        cv_scores_train.append(train_loss)
        cv_scores_val.append(val_loss)

        print(f"Fold {fold}: Training Log Loss = {train_loss:.4f}, Validation Log Loss = {val_loss:.4f}")

    avg_train_loss = np.mean(cv_scores_train)
    avg_val_loss = np.mean(cv_scores_val)

    print("\nTraining Log Loss scores for each fold:", cv_scores_train)
    print("Validation Log Loss scores for each fold:", cv_scores_val)
    print(f"Average Training Log Loss: {avg_train_loss:.4f}")
    print(f"Average Validation Log Loss: {avg_val_loss:.4f}")

    return avg_train_loss, avg_val_loss

# Grid search over reg_lambda and reg_alpha

reg_lambda_values = [1.0, 1.5, 2.0]
reg_alpha_values = [10.0, 12.0, 15.0]

best_config = None
best_val_loss = float('inf')

for reg_lambda in reg_lambda_values:
    for reg_alpha in reg_alpha_values:
        print(f"Testing reg_lambda={reg_lambda}, reg_alpha={reg_alpha}")
        avg_train_loss, avg_val_loss = cross_validate_xgboost(X, y, y_class_labels, n_splits=7, reg_lambda=reg_lambda, reg_alpha=reg_alpha)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_config = (reg_lambda, reg_alpha)

print(f"\nBest configuration: reg_lambda={best_config[0]}, reg_alpha={best_config[1]} with Validation Log Loss: {best_val_loss:.4f}")


Testing reg_lambda=1.0, reg_alpha=10.0
Fold 1: Training Log Loss = 1.3248, Validation Log Loss = 1.8912
Fold 2: Training Log Loss = 1.3253, Validation Log Loss = 1.9000
Fold 3: Training Log Loss = 1.3130, Validation Log Loss = 1.9599
Fold 4: Training Log Loss = 1.3295, Validation Log Loss = 1.9159
Fold 5: Training Log Loss = 1.3105, Validation Log Loss = 1.8852
Fold 6: Training Log Loss = 1.3164, Validation Log Loss = 1.8894
Fold 7: Training Log Loss = 1.3165, Validation Log Loss = 1.9120

Training Log Loss scores for each fold: [1.3247775477680879, 1.32531529102862, 1.3130167625590456, 1.329539947396641, 1.3105114406025944, 1.3163712668488143, 1.3165387301876537]
Validation Log Loss scores for each fold: [1.8912424078801582, 1.8999759179705245, 1.9598831550780187, 1.9159219489317196, 1.8852293742514097, 1.8893661713578196, 1.9119963630547259]
Average Training Log Loss: 1.3194
Average Validation Log Loss: 1.9077
Testing reg_lambda=1.0, reg_alpha=12.0
Fold 1: Training Log Loss = 1.4510,

In [52]:
from sklearn.model_selection import KFold
from xgboost import XGBClassifier
from sklearn.metrics import log_loss
import numpy as np

cv_scores_val = []  # Store validation scores
cv_scores_train = []  # Store training scores

kf = KFold(n_splits=7, shuffle=True, random_state=42)

for fold, (train_index, val_index) in enumerate(kf.split(X), 1):
    # Split data into training and validation sets
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]

    # Get class labels for training
    y_train = y_class_labels[train_index]

    # Get one-hot encoded labels for training and validation
    y_train_onehot = y.iloc[train_index].values
    y_val_onehot = y.iloc[val_index].values

    # Also get class labels for validation (needed for XGBoost)
    y_val_labels = y_class_labels[val_index]

    # Train XGBoost model
    xgb_model = XGBClassifier(
        objective='multi:softprob',  # Multiclass classification
        num_class=10,                # Number of classes
        eval_metric='mlogloss',      # Multiclass log loss
        random_state=42,
        reg_lambda=1.5,
        reg_alpha=12.0,
    )
    xgb_model.fit(X_train, y_train)

    # Predict probabilities for training and validation sets
    y_pred_proba_train = xgb_model.predict_proba(X_train)  # Training probabilities
    y_pred_proba_val = xgb_model.predict_proba(X_val)      # Validation probabilities

    # Compute multiclass log loss for training and validation
    train_loss = log_loss(y_train_onehot, y_pred_proba_train)
    val_loss = log_loss(y_val_onehot, y_pred_proba_val)

    # Append scores
    cv_scores_train.append(train_loss)
    cv_scores_val.append(val_loss)

    print(f"Fold {fold}: Training Log Loss = {train_loss:.4f}, Validation Log Loss = {val_loss:.4f}")

# Print results
print("\nTraining Log Loss scores for each fold:", cv_scores_train)
print("Validation Log Loss scores for each fold:", cv_scores_val)
print(f"Average Training Log Loss: {np.mean(cv_scores_train):.4f}")
print(f"Average Validation Log Loss: {np.mean(cv_scores_val):.4f}")


Fold 1: Training Log Loss = 1.4414, Validation Log Loss = 1.8901
Fold 2: Training Log Loss = 1.4539, Validation Log Loss = 1.9006
Fold 3: Training Log Loss = 1.4416, Validation Log Loss = 1.9621
Fold 4: Training Log Loss = 1.4486, Validation Log Loss = 1.9062
Fold 5: Training Log Loss = 1.4351, Validation Log Loss = 1.8737
Fold 6: Training Log Loss = 1.4487, Validation Log Loss = 1.8911
Fold 7: Training Log Loss = 1.4466, Validation Log Loss = 1.9049

Training Log Loss scores for each fold: [1.4414236042534743, 1.4538561486526944, 1.441590221941695, 1.4486499752671007, 1.4350981939726473, 1.4487269850798463, 1.44664367573841]
Validation Log Loss scores for each fold: [1.8901430541451605, 1.9005598034749005, 1.9621336037182444, 1.906234427417319, 1.8737479295768598, 1.8910729961059254, 1.9049375128402868]
Average Training Log Loss: 1.4451
Average Validation Log Loss: 1.9041


In [ ]:
# feature_importances = pd.DataFrame()

# # Loop through each target's estimator in the multioutput model
# for i, estimator in enumerate(multioutput_model.estimators_):
#     importance = estimator.feature_importances_
#     feature_importances[f"Target_{i+1}"] = importance

# # Average feature importance across all target columns
# feature_importances['Average_Importance'] = feature_importances.mean(axis=1)

# # Add feature names for reference
# feature_importances['Feature'] = X.columns

# # Sort features by average importance
# feature_importances = feature_importances.sort_values(by='Average_Importance', ascending=False)


In [37]:
# from sklearn.model_selection import KFold
# from xgboost import XGBClassifier
# from sklearn.metrics import log_loss
# import numpy as np

# cv_scores = []

# kf = KFold(n_splits=7, shuffle=True, random_state=42)

# for train_index, val_index in kf.split(X):
#     # Split data into training and validation sets
#     X_train, X_val = X.iloc[train_index], X.iloc[val_index]
#     y_train, y_val = y.iloc[train_index], y.iloc[val_index]

#     # Train XGBoost model
#     xgb_model = XGBClassifier(
#         objective='multi:softprob',  # Multiclass classification
#         num_class=10,                # Number of classes
#         eval_metric='mlogloss',      # Multiclass log loss
#         # use_label_encoder=False,     # Avoid warnings
#         random_state=42
#     )
#     xgb_model.fit(X_train, y_train)

#     # Predict probabilities
#     y_pred_proba = xgb_model.predict_proba(X_val)  # Shape: (n_samples, num_classes)

#     # Compute multiclass log loss
#     fold_loss = log_loss(y_val, y_pred_proba, labels=list(range(10)))

#     cv_scores.append(fold_loss)

# # Print results
# print("Log Loss scores for each fold:", cv_scores)
# print("Average Log Loss:", np.mean(cv_scores))


In [ ]:
merge_test = merged_data[(merged_data['source_hh'] == 'test') | (merged_data['source_edu'] == 'test')]
psu_hh_id_merge_test = merge_test['psu_hh_idcode']
merge_test = merge_test.drop(columns=['psu_hh_idcode', 'source_hh', 'birth_date_father', 'birth_date_mother', 'birth_date', 'source_edu', 'birth_date_spouse', 'hhid'])

In [56]:
for df in [education_train, education_test]:
    if 'psu_hh_idcode' not in df.columns:
        df['psu_hh_idcode'] = df['psu'].astype(str) + "_" + df['hh'].astype(str) + "_" + df['idcode'].astype(str)
        df.drop(columns=['psu', 'hh', 'idcode'], inplace=True)  # Remove individual columns after creating psu_hh_idcode

for df in [household_train, household_test]:
    if 'psu_hh_idcode' not in df.columns:
        df['psu_hh_idcode'] = df['psu'].astype(str) + "_" + df['hh'].astype(str) + "_" + df['idcode'].astype(str)
        df.drop(columns=['psu', 'hh', 'idcode'], inplace=True)  # Remove individual columns after creating psu_hh_idcode

education_train['source'] = 'train'
education_test['source'] = 'test'
household_train['source'] = 'train'
household_test['source'] = 'test'

education_combined = pd.concat([education_train, education_test], axis=0).reset_index(drop=True)
household_combined = pd.concat([household_train, household_test], axis=0).reset_index(drop=True)

education_combined = education_combined.rename(columns={col: col + '_edu' for col in education_train.columns if col != 'psu_hh_idcode'})
household_combined = household_combined.rename(columns={col: col + '_hh' for col in household_train.columns if col != 'psu_hh_idcode' and col != 'hhid'})

merged_data = pd.merge(education_combined, household_combined, on='psu_hh_idcode', how='outer')
merged_data = pd.merge(merged_data, subjective_poverty_train, on='psu_hh_idcode', how='outer')

col_names = {"q06_hh": "marital_status", 
             "q07_hh": "spouse_live", 
             "q08_hh": "spouse_id", 
             "q11_hh": "mother_live", 
             "q12_hh": "mother_id", 
             "q13_hh": "mother_education", 
             "q18_hh": "father_id", 
             "q19_hh": "father_education",
             'q03_edu': "has_attended_school", 
             'q04_edu': "highest_grade_completed1",
             'q05_edu': "highest_grade_completed2",
             'q06_edu': "highest_diploma",
             'q07_edu': "no_preschool_years",
             'Q10_edu': "reason_not_going_school",
             'q01_edu': "can_read",
             'q02_edu': "can_write",
             'Q30_edu': "transport_subsidy_received",
             'Q31_edu': "transport_subsidy_received_amount",
             'Q41_edu': "spent_on_education",
             'Q47_edu': "value_textbook_subsidy",
             'Q50_edu': "private_tutoring",
             'Q53_edu': "times_private_tutor",
             'Q56_edu': "tutoring_pay",
             'Q64_edu': "scholarship",
             'Q65_edu': "scholarship_value"
             }

merged_data = merged_data.rename(columns=col_names)
merged_data[['psu', 'hh', 'idcode']] = merged_data['psu_hh_idcode'].str.split('_', expand=True).astype(int)



In [ ]:
# Pre-group data for faster lookups
merged_data_dict = merged_data.set_index(['psu', 'hh', 'idcode']).to_dict(orient='index')

def fetch_relative_data_optimized(row, relative_id_column, suffix):
    if not pd.isna(row[relative_id_column]):
        relative_id = int(row[relative_id_column])
        key = (row['psu'], row['hh'], relative_id)
        
        if key in merged_data_dict:
            relative_data = merged_data_dict[key]
            for col, value in relative_data.items():
                if col not in ['psu', 'hh', 'idcode', 'psu_hh_idcode']:  # Exclude redundant columns
                    row[f"{col}_{suffix}"] = value
    return row

# Apply the optimized function to the DataFrame
for relative, suffix in [('spouse_id', 'spouse'), ('mother_id', 'mother'), ('father_id', 'father')]:
    merged_data = merged_data.apply(fetch_relative_data_optimized, axis=1, args=(relative, suffix))
